# 🧠 AI Logic Engine: Main Inference Console

Firestore ထဲရှိ Logic Graph ကို ရှာဖွေပြီး Concept နှစ်ခုအကြား ဆက်နွယ်မှုကို ထုတ်ပေးသည်။

In [ ]:
# Step 1: Initialize
!pip install -q firebase-admin

from google.colab import files
import firebase_admin
from firebase_admin import credentials, firestore
import re

DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2'
gcp_credentials = None

if not firebase_admin._apps:
    print("Please upload your serviceAccountKey.json file:")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("❌ JSON key လိုအပ်ပါသည်။")
    filename = list(uploaded.keys())[0]
    from google.oauth2 import service_account
    global gcp_credentials
    gcp_credentials = service_account.Credentials.from_service_account_file(filename)
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

from google.cloud import firestore as g_firestore
project_id = firebase_admin.get_app().project_id
db = g_firestore.Client(project=project_id, database=DATABASE_ID, credentials=gcp_credentials)
print(f"✅ Memory Node Synced: {DATABASE_ID}")

In [ ]:
# Step 2: Symbolic Logic Core
class KnowledgeInferenceEngine:
    def __init__(self, db):
        self.db = db
        self.cache = {}

    def clean_id(self, text):
        import hashlib
        if not text: return ""
        text = str(text).lower().strip()
        if re.search(r'[^\x00-\x7F]', text):
            return 'my_' + hashlib.md5(text.encode('utf-8')).hexdigest()[:16]
        text = re.sub(r'[\s/\\.#\[\]\*\?\!]+', '_', text)
        text = re.sub(r'_{2,}', '_', text)
        return text.strip('_')[:128]

    def _get_node(self, node_id):
        if node_id in self.cache:
            return self.cache[node_id]
        doc = self.db.collection('nodes').document(node_id).get()
        if doc.exists:
            self.cache[node_id] = doc.to_dict()
            return self.cache[node_id]
        return None

    def solve(self, start, target, depth=10):
        sid, tid = self.clean_id(start), self.clean_id(target)
        if sid == tid: return [f"{start} corresponds directly to {target}."]
        
        queue = [(sid, [])]
        visited = {sid}
        
        while queue:
            curr_id, path = queue.pop(0)
            
            if curr_id == tid: return path
            if len(path) >= depth: continue
            
            # API Read with Cache
            data = self._get_node(curr_id)
            if not data: continue
            
            # Path 1: Groups (Inheritance)
            for g in data.get('groups', []):
                g_id = self.clean_id(g)
                if g_id not in visited:
                    visited.add(g_id)
                    queue.append((g_id, path + [f"{curr_id} is a type of {g}"]))
            
            # Path 2: Relations
            for rel in data.get('relations', []):
                target_ptr = rel.get('targetId')
                if target_ptr:
                    t_id = self.clean_id(target_ptr)
                    if t_id not in visited:
                        visited.add(t_id)
                        sentence = rel.get('sentence') or f"{curr_id} {rel['verb'].replace('_',' ')} {target_ptr}"
                        queue.append((t_id, path + [sentence]))
                    
        return None

engine = KnowledgeInferenceEngine(db)
print("✅ Inference Engine Ready.")

In [ ]:
# @title 🔍 Step 3: Run Inference Proof
CONCEPT_A = "Mitochondria" # @param {type:"string"}
CONCEPT_B = "Cellular Energy" # @param {type:"string"}
LIMIT = 10 # @param {type:"slider", min:1, max:20, step:1}

print(f"🧠 Calculating Proof: '{CONCEPT_A}' ➔ '{CONCEPT_B}'...\n")
proof = engine.solve(CONCEPT_A, CONCEPT_B, depth=LIMIT)

if proof:
    print(f"✅ Connection Verified:\n")
    for i, step in enumerate(proof):
        print(f"  [{i+1}] {step}")
else:
    print("❌ Path not found. Hint: Run ingestion longer.")